# Prototype Run Notebook

This notebook runs inference for a selected model stage and exports `predictions.csv`.

## What this notebook does
- Selects a model stage: Baseline, Stage 1, Stage 2, or Stage 3
- Loads one audio file or a folder of audio files
- Runs transcription
- Saves outputs in a timestamped folder

In [ ]:
!pip -q install transformers peft librosa soundfile pandas torch

In [ ]:
from pathlib import Path
import argparse
import json
from datetime import datetime

import pandas as pd
import torch
import librosa
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from peft import PeftModel

In [ ]:
# -----------------------------
# CONFIG
# -----------------------------
STAGE = "stage3"   # baseline | stage1 | stage2 | stage3
INPUT_PATH = "/content/sample_inputs"   # file or folder
OUTPUT_DIR = "/content/outputs"

MODEL_NAME = "openai/whisper-small.en"
STAGE1_ADAPTER = "/content/models/stage1_adapter"
STAGE2_ADAPTER = "/content/models/stage2_adapter"
STAGE3_ADAPTER = "/content/models/stage3_adapter" 

In [ ]:
def set_generation_defaults(model):
    model.generation_config.forced_decoder_ids = None
    model.generation_config.suppress_tokens = []
    return model


def load_model(stage, model_name, device):
    if stage == "baseline":
        model = WhisperForConditionalGeneration.from_pretrained(model_name).to(device)
        return set_generation_defaults(model)

    adapter_map = {
        "stage1": STAGE1_ADAPTER,
        "stage2": STAGE2_ADAPTER,
        "stage3": STAGE3_ADAPTER,
    }
    adapter_dir = adapter_map[stage]

    base = WhisperForConditionalGeneration.from_pretrained(model_name).to(device)
    base = set_generation_defaults(base)
    model = PeftModel.from_pretrained(base, adapter_dir).to(device)
    return set_generation_defaults(model)


@torch.no_grad()
def transcribe_file(model, processor, wav_path, device, max_new_tokens=32):
    y, _ = librosa.load(wav_path, sr=16000)
    inp = processor(y, sampling_rate=16000, return_tensors="pt").to(device)
    gen = model.generate(inp.input_features, max_new_tokens=max_new_tokens, num_beams=1, do_sample=False)
    return processor.batch_decode(gen, skip_special_tokens=True)[0].strip()


def collect_audio_files(input_path):
    input_path = Path(input_path)
    if input_path.is_file():
        return [input_path]
    exts = {".wav", ".mp3", ".flac", ".m4a"}
    return sorted([p for p in input_path.rglob("*") if p.suffix.lower() in exts])

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_dir = Path(OUTPUT_DIR) / f"run_{timestamp}_{STAGE}"
run_dir.mkdir(parents=True, exist_ok=True)

processor = WhisperProcessor.from_pretrained(MODEL_NAME)
model = load_model(STAGE, MODEL_NAME, device)

audio_files = collect_audio_files(INPUT_PATH)
if not audio_files:
    raise FileNotFoundError("No audio files found in INPUT_PATH")

rows = []
for wav_path in audio_files:
    pred = transcribe_file(model, processor, str(wav_path), device)
    rows.append({
        "audio_path": str(wav_path),
        "hyp_text": pred,
        "model_stage": STAGE,
        "timestamp": timestamp,
    })

pred_df = pd.DataFrame(rows)
pred_df.to_csv(run_dir / "predictions.csv", index=False)

with open(run_dir / "run_config.json", "w", encoding="utf-8") as f:
    json.dump({"stage": STAGE, "input_path": INPUT_PATH, "model_name": MODEL_NAME}, f, indent=2)

print("Saved predictions to:", run_dir / "predictions.csv")
pred_df.head()